In [2]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")  # 2.4M 파라미터, COCO 80 class
results = model.predict("mujoco_objects.png", conf=0.5)

# 결과 구조
for box in results[0].boxes:
    cls_id = int(box.cls)             # 클래스 ID (0=person, 56=chair, ...)
    cls_name = model.names[cls_id]    # "person", "chair", ...
    conf = float(box.conf)            # 신뢰도 (0.0~1.0)
    x, y, w, h = box.xywh[0].tolist() # 중심 좌표 + 크기 (px)
    print(f"{cls_name}: conf={conf:.2f}, center=({x:.0f},{y:.0f}), size=({w:.0f}x{h:.0f})")


image 1/1 /home/user/source/python312/python/VLA/mujoco_objects.png: 480x640 1 bottle, 14.9ms
Speed: 1.5ms preprocess, 14.9ms inference, 10.2ms postprocess per image at shape (1, 3, 480, 640)
bottle: conf=0.58, center=(37,284), size=(56x156)


In [5]:
system_prompt = """너는 모바일 로봇의 시각 분석가이다.
첨부된 이미지를 분석하여, 로봇이 안전하게 이동하기 위해 알아야 할 정보를 JSON으로 반환하라.

출력 형식 (반드시 유효한 JSON만 출력):
{
  "scene_summary": "장면 한 줄 요약 (한국어)",
  "social_hints": [
    {
      "type": "avoid_between_people",
      "reason": "이유 (영어, 짧게)",
      "confidence": 0.0~1.0,
      ...
    }
  ]
}

type 필드는 반드시 avoid_between_people, prefer_side_pass, slow_down, clear_path 중 하나의 문자열만 사용하라.```json은 적지마라. """

In [11]:
import base64
from openai import OpenAI

# Ollama 연결 (Ch03과 동일한 패턴)
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

# 이미지를 base64로 인코딩
with open("apple_ex.jpg", "rb") as f:
    b64 = base64.b64encode(f.read()).decode()

# VLM 호출
response = client.chat.completions.create(
    # model="qwen3.6:27b",
    model="gemma4:e4b",
    reasoning_effort="none",
    # stream=True,
    # temperature=0,
    # max_tokens=512,
    messages=[
        {"role": "system", "content": system_prompt},
        {
        "role": "user",
        "content": [
            {"type": "text", "text": "이 장면을 JSON으로 분석해주세요."},
            {"type": "image_url",
             "image_url": {"url": f"data:image/jpeg;base64,{b64}"}}
        ]
    }],
)

print(response.choices[0].message.content)

{
  "scene_summary": "배경이 단순한 환경에서 한 사람이 사과 두 개를 들고 포즈를 취하고 있다.",
  "social_hints": [
    {
      "type": "clear_path",
      "reason": "The path is clear, but the robot should move slowly to maintain safety.",
      "confidence": 0.9
    }
  ]
}


In [ ]:
# 시스템 프롬프트 적용 + 스트리밍 응답 시간 테스트
import base64
import time
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

with open("apple_ex.jpg", "rb") as f:
    b64 = base64.b64encode(f.read()).decode()

start = time.perf_counter()
first_chunk_at = None
first_content_at = None
received_text = []

stream = client.chat.completions.create(
    model="qwen3.6:27b",
    reasoning_effort="none",
    stream=True,
    temperature=0,
    max_tokens=512,
    messages=[
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "이 장면을 JSON으로 분석해주세요."},
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{b64}"}
                }
            ]
        }
    ],
)

print("[stream start]", flush=True)

for chunk in stream:
    now = time.perf_counter()
    if first_chunk_at is None:
        first_chunk_at = now
        print(f"[first chunk: {first_chunk_at - start:.2f}s]", flush=True)

    delta = chunk.choices[0].delta
    delta_data = delta.model_dump(exclude_none=True)

    text = delta_data.get("content")
    reasoning = (
        delta_data.get("reasoning_content")
        or delta_data.get("reasoning")
        or delta_data.get("thinking")
    )

    if reasoning:
        print(f"\n[reasoning chunk received: {now - start:.2f}s]", flush=True)
        continue

    if text:
        if first_content_at is None:
            first_content_at = now
            print(f"[first content: {first_content_at - start:.2f}s]\n", flush=True)
        received_text.append(text)
        print(text, end="", flush=True)

total = time.perf_counter() - start
print(f"\n\n[total: {total:.2f}s]", flush=True)

if first_content_at is None:
    print("[content 없음: reasoning 필드로만 왔는지 또는 빈 응답인지 확인 필요]", flush=True)


[stream start]
[first chunk: 9.23s]
[first content: 9.23s]

{
  "scene_summary": "주황색 배경 앞에 선 한 여성이 양손에 사과를 들고 비교하고 있습니다.",
  "social_hints": [
    {
      "type": "clear_path",
      "reason": "Single person standing still, no immediate crowd navigation needed",
      "confidence": 0.95
    }
  ]
}

[total: 16.68s]
